In [2]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

FEATURES = [
    "ret_1","ret_5","ret_10",
    "vol_5","vol_10","vol_20",
    "hl_pct","oc_pct",
    "log_volume",
    "daily_sentiment","daily_news_count","news_shock",
    "sent_3","sent_7",
    "quarterly_sentiment","annual_sentiment"
]

HORIZON_DAYS = 5   # or 10 (must match the model/scaler file you saved)
LOOKBACK = 60      # must match training lookback

# 1) Load model + scaler
model = load_model(f"bilstm_return_regression_{HORIZON_DAYS}d_final.keras")
scaler = joblib.load(f"feature_scaler_{HORIZON_DAYS}d.pkl")

# 2) Pick the dataframe you want to predict on
#    df_new must already contain all FEATURES columns computed the SAME way as training
df_new = df.copy()   # replace df with your new dataframe if you have one

# (Optional but strongly recommended) drop rows with missing features
df_new = df_new.dropna(subset=FEATURES).reset_index(drop=True)

# 3) Build new_X_raw  ✅ this fixes your NameError
new_X_raw = df_new[FEATURES].values.astype(np.float32)

# 4) Scale with the SAME scaler used in training
new_X_scaled = scaler.transform(new_X_raw)

# 5) Create sequences/windows
X_seq = np.array([new_X_scaled[i-LOOKBACK:i] for i in range(LOOKBACK, len(new_X_scaled))], dtype=np.float32)

# 6) Predict
pred = model.predict(X_seq, verbose=0).flatten()

print("Pred shape:", pred.shape)
print("First 10 predictions:", pred[:10])

# 7) Attach predictions back to df_new aligned correctly
df_new[f"pred_target_{HORIZON_DAYS}d"] = np.nan
df_new.loc[LOOKBACK:, f"pred_target_{HORIZON_DAYS}d"] = pred

df_new[[f"pred_target_{HORIZON_DAYS}d"]].tail()


NameError: name 'df' is not defined